# 🧹 02 — Data Cleaning

**Objective**: Handle missing values, fix data types, detect outliers, remove duplicates, and create a clean merged dataset ready for analysis.

**Why this step matters**: Dirty data leads to misleading analysis and poor ML models. Missing values bias aggregations, wrong data types prevent date calculations, and outliers can dominate model training.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_all, build_master_df
from src.data_cleaner import (
    analyze_missing, fix_date_columns, handle_missing_values,
    detect_outliers_iqr, cap_outliers, remove_duplicates,
    filter_delivered_orders
)

pd.set_option('display.max_columns', 50)
print("Libraries loaded ✓")

## 1. Load and Merge Data

In [ ]:
# Load all tables
dfs = load_all()

# Merge into master DataFrame
master = build_master_df(dfs)
print(f"\nMaster shape before cleaning: {master.shape}")
master.head()

## 2. Missing Value Analysis

In [ ]:
# Check missing values across all columns
missing_report = analyze_missing(master)
print("\nColumns with missing values:")
missing_report

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(10, 5))
if len(missing_report) > 0:
    ax.barh(missing_report['column'], missing_report['missing_pct'],
            color='#ef4444', edgecolor='none', height=0.6)
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column', fontweight='bold')
    ax.invert_yaxis()
    for i, v in enumerate(missing_report['missing_pct']):
        ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=10)
else:
    ax.text(0.5, 0.5, 'No missing values!', ha='center', va='center',
            fontsize=16, transform=ax.transAxes)
plt.tight_layout()
plt.show()

## 3. Fix Data Types

Date columns are stored as strings — we need to convert them to `datetime` for time-based calculations.

In [ ]:
# Convert date columns
master = fix_date_columns(master)
print("\nDate column types after fix:")
for col in [c for c in master.columns if 'date' in c or 'timestamp' in c]:
    print(f"  {col}: {master[col].dtype}")

## 4. Handle Missing Values

In [ ]:
# Apply missing value strategies
master = handle_missing_values(master)

# Verify
remaining = analyze_missing(master)
print(f"\nRemaining columns with missing values: {len(remaining)}")
if len(remaining) > 0:
    print(remaining)

## 5. Outlier Detection

E-commerce order values are naturally right-skewed (many small orders, few large ones). We use the **IQR method** to detect and cap extreme values.

In [ ]:
# Check outliers in price
print("PRICE OUTLIERS:")
price_outliers = detect_outliers_iqr(master, 'price', factor=1.5)

print("\nPAYMENT VALUE OUTLIERS:")
payment_outliers = detect_outliers_iqr(master, 'payment_value', factor=1.5)

In [ ]:
# Visualize price distribution before capping
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(master['price'].dropna(), bins=50, color='#6366f1',
             edgecolor='none', alpha=0.8)
axes[0].set_title('Price Distribution (Before Capping)', fontweight='bold')
axes[0].set_xlabel('Price (R$)')
axes[0].axvline(master['price'].quantile(0.75) + 1.5 *
                (master['price'].quantile(0.75) - master['price'].quantile(0.25)),
                color='#ef4444', linestyle='--', label='Upper IQR bound')
axes[0].legend()

# Cap outliers
master = cap_outliers(master, 'price', factor=1.5)

axes[1].hist(master['price'].dropna(), bins=50, color='#10b981',
             edgecolor='none', alpha=0.8)
axes[1].set_title('Price Distribution (After Capping)', fontweight='bold')
axes[1].set_xlabel('Price (R$)')

plt.tight_layout()
plt.show()

## 6. Remove Duplicates

In [ ]:
master = remove_duplicates(master)

## 7. Filter to Delivered Orders

In [ ]:
# Keep only delivered orders for analysis
master_clean = filter_delivered_orders(master)

## 8. Save Cleaned Dataset

In [ ]:
# Save for use in subsequent notebooks
master_clean.to_csv('../data/master_cleaned.csv', index=False)
print(f"\n✓ Saved cleaned dataset: {master_clean.shape[0]:,} rows × {master_clean.shape[1]} columns")
print(f"  File: data/master_cleaned.csv")

## Summary

| Step | Action | Impact |
|------|--------|--------|
| Missing values | Filled review_score with median, categories with 'other' | No more NaN in key analysis columns |
| Data types | Converted 5 date columns to datetime | Enables time-based calculations |
| Outliers | Capped prices using IQR method | Reduces extreme value influence on models |
| Duplicates | Checked and removed any duplicate rows | Prevents double-counting |
| Filtering | Kept only delivered orders | Clean revenue/delivery metrics |

**Next Step**: In notebook 03, we'll perform Exploratory Data Analysis on this cleaned dataset.